[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Connections and Cursors &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/stations.db` with the year of readings, as the notebook's Setup did.
Run it first. The tasks do not depend on one another, and the last cell removes the scratch folder.


In [1]:
import math
import shutil
import sqlite3
from contextlib import closing
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "stations.db"
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}


def year_of_readings():
    """Every hour of 2025 at the four stations, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


DATABASE.unlink(missing_ok=True)
with closing(sqlite3.connect(DATABASE)) as conn, conn:
    conn.execute("CREATE TABLE readings (station TEXT NOT NULL, hour TEXT NOT NULL, celsius REAL)")
    conn.executemany("INSERT INTO readings (station, hour, celsius) VALUES (?, ?, ?)", year_of_readings())

print("built", DATABASE)


built scratch/stations.db


**1.** Different stations, through `closing`.


In [2]:
with closing(sqlite3.connect(DATABASE)) as conn:
    (stations,) = conn.execute("SELECT COUNT(DISTINCT station) FROM readings").fetchone()

print("stations:", stations)


stations: 4


`COUNT(DISTINCT station)` counts each station name once. The row it returns is a tuple of one value,
which `(stations,) =` unpacks.


**2.** One row at a time, then two.


In [3]:
with closing(sqlite3.connect(DATABASE)) as conn:
    cursor = conn.execute("SELECT hour, celsius FROM readings WHERE station = ? ORDER BY hour", ("Tromso",))
    print(cursor.fetchone())
    print(cursor.fetchone())
    print(cursor.fetchone())
    print(cursor.fetchmany(2))
    cursor.close()


('2025-01-01T00:00', -6.7)
('2025-01-01T01:00', -8.5)
('2025-01-01T02:00', -8.5)
[('2025-01-01T03:00', -8.3), ('2025-01-01T04:00', -7.9)]


Every fetch takes up where the one before stopped, so `fetchmany(2)` hands back the fourth and fifth
hours of the year. The cursor stops partway through Tromso's year, so it is closed before the block
closes the connection, since a cursor left half read keeps the database locked.


**3.** `fetchmany` and `arraysize`.


In [4]:
with closing(sqlite3.connect(DATABASE)) as conn:
    cursor = conn.execute("SELECT hour, celsius FROM readings WHERE station = ? ORDER BY hour", ("Svalbard",))
    print("arraysize", cursor.arraysize, "->", cursor.fetchmany())
    cursor.arraysize = 4
    print("arraysize", cursor.arraysize, "->", cursor.fetchmany())
    cursor.close()


arraysize 1 -> [('2025-01-01T00:00', -14.6)]
arraysize 4 -> [('2025-01-01T01:00', -16.4), ('2025-01-01T02:00', -16.4), ('2025-01-01T03:00', -16.2), ('2025-01-01T04:00', -15.8)]


With no number, `fetchmany` takes `arraysize` rows, which starts at 1. Setting `arraysize` changes
what every later `fetchmany()` on that cursor takes.


**4.** Column names before any row.


In [5]:
with closing(sqlite3.connect(DATABASE)) as conn:
    cursor = conn.execute(
        "SELECT station, MIN(celsius) AS coldest, MAX(celsius) AS warmest FROM readings GROUP BY station"
    )
    print([column[0] for column in cursor.description])


['station', 'coldest', 'warmest']


`description` is filled in as soon as the statement runs. The names come from the `AS` aliases, and a
column without one, such as `station`, keeps the name it was selected by.


**5.** A function that closes what it opens.


In [6]:
def first_hour_below(database, station, celsius):
    """The first hour a station's reading fell below a temperature, or None if it never did."""
    with closing(sqlite3.connect(database)) as conn:
        row = conn.execute(
            "SELECT hour FROM readings WHERE station = ? AND celsius < ? ORDER BY hour LIMIT 1", (station, celsius)
        ).fetchone()
    return row[0] if row else None


print("Oslo below -5:", first_hour_below(DATABASE, "Oslo", -5))
print("Bergen below -20:", first_hour_below(DATABASE, "Bergen", -20))


Oslo below -5: 2025-01-01T01:00
Bergen below -20: None


`fetchone` runs inside the block, while the connection is open, and the function returns a plain
string, or `None` when `fetchone` found no row, so nothing it returns depends on the closed
connection.


**6.** The biggest rise from one hour to the next.


In [7]:
biggest_rise, where = 0.0, None
previous = {}

with closing(sqlite3.connect(f"file:{DATABASE}?mode=ro", uri=True)) as conn:
    for station, hour, celsius in conn.execute("SELECT station, hour, celsius FROM readings ORDER BY station, hour"):
        before = previous.get(station)
        if celsius is not None and before is not None and celsius - before > biggest_rise:
            biggest_rise, where = celsius - before, (station, hour)
        previous[station] = celsius

print("biggest rise:", round(biggest_rise, 1), "at", where)


biggest rise: 1.1 at ('Bergen', '2025-05-23T10:00')


The loop keeps each station's previous reading in a dictionary rather than keeping the rows. A
missing reading is stored as `None` in `previous`, so the hour after it is not compared with
anything, since nobody knows how far the temperature moved.

Last, remove the scratch folder:


In [8]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Connections and Cursors](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/02-connections-and-cursors.ipynb)  &nbsp;&middot;&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)
